In [ ]:
# === Setup: imports and environment ===
import os
from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter, SpacyTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain
from langchain.retrievers import BM25Retriever, EnsembleRetriever

load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

# Models
EMBED_MODEL = "text-embedding-3-small"
LLM_MODEL = "gpt-3.5-turbo"

## 1) Multi-PDF Support (with source tracking)
Steps:
1. Load multiple PDFs.
2. Tag each Document with source filename.
3. Combine docs; build vector store.
4. Filter by source if needed.

In [ ]:
from pathlib import Path

def load_pdfs(pdf_paths):
    """Load PDFs and tag each page with source_file."""
    docs_all = []
    for path in pdf_paths:
        loader = PyPDFLoader(str(path))
        docs = loader.load()
        for d in docs:
            d.metadata["source_file"] = Path(path).name
        docs_all.extend(docs)
    return docs_all

# Example paths (replace with your PDFs)
pdf_paths = [
    "./HDFC-Life-Group-Term-Life-Policy.pdf",
    "./HDFC-Life-Sanchay-Plus-Life-Long-Income-Option-101N134V19-Policy-Document.pdf",
    "./HDFC-Life-Sampoorna-Jeevan-101N158V04-Policy-Document (1).pdf",
]

documents = load_pdfs(pdf_paths)
print(f"Loaded {len(documents)} pages from {len(pdf_paths)} PDFs")

## 2) Semantic Chunking (variable, structure-aware)
Use spaCy-based splitter to respect sentences/paragraphs; fall back to recursive splitter for flexibility.

In [ ]:
# Semantic splitter (spaCy). Requires 'en_core_web_sm' installed. If not, pip install spacy && python -m spacy download en_core_web_sm
semantic_splitter = SpacyTextSplitter(chunk_size=1000, chunk_overlap=150)

# Fallback recursive splitter for robustness
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", " ", ""],
)

# Choose which to use
use_semantic = True
splitter = semantic_splitter if use_semantic else recursive_splitter

chunks = splitter.split_documents(documents)
print(f"Created {len(chunks)} chunks; avg length = {sum(len(c.page_content) for c in chunks)/len(chunks):.0f} chars")

## 3) Build Vector Store (FAISS)

In [ ]:
embeddings = OpenAIEmbeddings(model=EMBED_MODEL)
vectorstore = FAISS.from_documents(chunks, embeddings)
print("Vector store ready with", len(chunks), "chunks")

## 4) Citation System
- Prompt enforces citations.
- Each chunk already has source_file and page metadata from PyPDFLoader.

In [ ]:
# Retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

# Prompt that demands citations
system_prompt = """You are a PDF QA assistant. Use ONLY the context to answer.
Cite sources inline like (source_file, page X). If unsure, say you don't know.

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{question}"),
])

llm = ChatOpenAI(model=LLM_MODEL, temperature=0)

# Helper to format docs with citations

def format_docs(docs):
    formatted = []
    for d in docs:
        src = d.metadata.get("source_file", "pdf")
        page = d.metadata.get("page", "?")
        formatted.append(f"(Source: {src}, page {page})\n{d.page_content}")
    return "\n\n".join(formatted)

rag_chain_citation = (
    {"context": retriever | format_docs, "question": lambda x: x["question"]}
    | prompt
    | llm
    | StrOutputParser()
)

# Test
question = "What are the main policy benefits?"
answer = rag_chain_citation.invoke({"question": question})
print(answer)

## 5) Conversational Memory (follow-ups handled)
Use ConversationalRetrievalChain with ConversationBufferMemory.

In [ ]:
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)
conv_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory,
    combine_docs_chain_kwargs={"document_prompt": None},
)

# Demo conversation
q1 = "Summarize the policy benefits."  # initial
q2 = "And what about exclusions?"      # follow-up

print("Q1:", q1)
print(conv_chain.invoke({"question": q1})["answer"], "\n")
print("Q2:", q2)
print(conv_chain.invoke({"question": q2})["answer"])

## 6) Streamlit UI (upload + chat)
Copy this cell into a `streamlit_app.py` and run `streamlit run streamlit_app.py`.

In [ ]:
streamlit_code = r'''
import os
from pathlib import Path
import streamlit as st
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory

load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

st.set_page_config(page_title="PDF Chat", page_icon="📄", layout="wide")
st.title("📄 PDF RAG Chat")

uploaded_files = st.file_uploader("Upload PDFs", type=["pdf"], accept_multiple_files=True)
if uploaded_files:
    docs = []
    for uf in uploaded_files:
        tmp_path = Path(uf.name)
        tmp_path.write_bytes(uf.read())
        loader = PyPDFLoader(str(tmp_path))
        d = loader.load()
        for page in d:
            page.metadata["source_file"] = uf.name
        docs.extend(d)
    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    chunks = splitter.split_documents(docs)
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
    vs = FAISS.from_documents(chunks, embeddings)
    retriever = vs.as_retriever(search_kwargs={"k": 4})
    llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
    memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)
    chain = ConversationalRetrievalChain.from_llm(llm, retriever, memory=memory)

    st.success(f"Indexed {len(chunks)} chunks from {len(uploaded_files)} PDFs")
    user_q = st.chat_input("Ask about your PDFs")
    if user_q:
        result = chain.invoke({"question": user_q})
        st.chat_message("assistant").write(result["answer"])
'''
print(streamlit_code)

## 7) Advanced Retrieval (Hybrid: Vector + BM25 + MMR)
Combine FAISS (dense) with BM25 (sparse) using EnsembleRetriever.

In [ ]:
# Build BM25 retriever on the same chunks
bm25 = BM25Retriever.from_documents(chunks)

# Optional: enable MMR by setting search_type and search_kwargs on vector retriever
vector_mmr = vectorstore.as_retriever(search_type="mmr", search_kwargs={"k": 6, "lambda_mult": 0.5})

hybrid_retriever = EnsembleRetriever(
    retrievers=[vector_mmr, bm25],
    weights=[0.6, 0.4],
)

# Hybrid RAG chain
hybrid_prompt = ChatPromptTemplate.from_messages([
    ("system", "Use the context to answer. Cite sources. If unsure, say you don't know.\n\n{context}"),
    ("human", "{question}"),
])

def format_docs_hybrid(docs):
    return "\n\n".join(
        f"(Source: {d.metadata.get('source_file','pdf')}, page {d.metadata.get('page','?')})\n{d.page_content}"
        for d in docs
    )

hybrid_chain = (
    {"context": hybrid_retriever | format_docs_hybrid, "question": lambda x: x["question"]}
    | hybrid_prompt
    | llm
    | StrOutputParser()
)

print(hybrid_chain.invoke({"question": "List the major exclusions."}))

## 8) Evaluation Pointers (manual + quick heuristics)
- Check retrieval quality: low distance / relevant text.
- Use `similarity_search_with_score` to inspect scores.
- Spot-check citations: pages mentioned should match content.
- Latency: measure end-to-end time per query.
- Cost: reuse saved FAISS stores; keep k modest (3-5).